# 🎬 03 · 视频多模态与音视频理解

> 视频 = 帧序列 + （可能）音频。本节讲**视频多模态的三种建模路线**（帧拼接/时间池化/时序注意力）、
> **音视频对齐**思想，并手写一个简化"帧序列 → 时序聚合 → 分类"流程。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 视频怎么喂给模型？ | 采样帧 → 帧级编码 → 时序建模 |
| 2 | 时序信息怎么聚合？ | 拼接 / 平均池化 / 时序注意力 |
| 3 | 音频怎么对进来？ | 音频 embedding 与视频对齐 |
| 4 | VideoLLM 怎么理解长视频？ | 关键帧 / 分段 / 记忆 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 视频建模三种路线（必背）

In [2]:
print('''路线1：帧拼接（early fusion）
  每帧过 ViT -> 帧 token 直接拼接成长序列 -> 过 LLM/Transformer
  优点：全信息；缺点：序列长、计算贵、难以长视频。

路线2：时间池化/平均（late fusion）
  每帧独立编码 -> 做帧平均/最大池化 -> 单向量
  优点：轻量；缺点：丢失时序关系（顺序打乱结果一样）。

路线3：时序注意力（推荐，VideoLLM 主流）
  帧 token 后加位置编码 -> 时序 Transformer / Q-Former
  优点：保留顺序关系、可控压缩；代表：Video-LLaMA、Qwen2-VL。
''')

路线1：帧拼接（early fusion）
  每帧过 ViT -> 帧 token 直接拼接成长序列 -> 过 LLM/Transformer
  优点：全信息；缺点：序列长、计算贵、难以长视频。

路线2：时间池化/平均（late fusion）
  每帧独立编码 -> 做帧平均/最大池化 -> 单向量
  优点：轻量；缺点：丢失时序关系（顺序打乱结果一样）。

路线3：时序注意力（推荐，VideoLLM 主流）
  帧 token 后加位置编码 -> 时序 Transformer / Q-Former
  优点：保留顺序关系、可控压缩；代表：Video-LLaMA、Qwen2-VL。



## 2️⃣ 手写：视频帧采样与帧级编码

In [3]:
# 模拟：30 秒视频，每 2 秒采 1 帧 = 15 帧，每帧由 ViT 编码为 1024 维
n_frames = 15
frame_feat = rng.normal(0, 1, (n_frames, 1024))
print('帧特征:', frame_feat.shape, '（15 帧 × 1024 维）')
print('➡️ 实际做法：抽帧（均匀/关键帧）+ 每帧过 ViT 得到 token 序列')

帧特征: (15, 1024) （15 帧 × 1024 维）
➡️ 实际做法：抽帧（均匀/关键帧）+ 每帧过 ViT 得到 token 序列


## 3️⃣ 三种时序聚合对比（含可视化）

In [4]:
# 帧位置编码（让模型知道顺序）
pos = np.arange(n_frames)[:, None] / n_frames
frame_feat2 = frame_feat + rng.normal(0, 0.05, frame_feat.shape) * pos

# 路线2：平均池化
avg_pool = frame_feat2.mean(axis=0)
print('平均池化输出:', avg_pool.shape)

# 路线3：时序自注意力（简化）
def temporal_attention(x):
    Wq = rng.normal(0, 0.1, (1024, 128)); Wk = rng.normal(0, 0.1, (1024, 128))
    Wv = rng.normal(0, 0.1, (1024, 128))
    Q = x @ Wq; K = x @ Wk; V = x @ Wv
    scores = Q @ K.T / np.sqrt(128)
    attn = np.exp(scores - scores.max(axis=1, keepdims=True))
    attn = attn / attn.sum(axis=1, keepdims=True)
    return attn @ V, attn

attn_out, attn_mat = temporal_attention(frame_feat2)
print('时序注意力输出:', attn_out.shape)
assert attn_mat.shape == (n_frames, n_frames) and np.allclose(attn_mat.sum(axis=1), 1)
print('✅ 时序注意力行归一化正确')

plt.figure(figsize=(6, 5))
plt.imshow(attn_mat, cmap='plasma')
plt.xlabel('被关注的帧'); plt.ylabel('查询帧')
plt.title('帧间时序注意力（相邻帧关注更强为佳）')
plt.colorbar()
plt.show()
print('✅ 三种聚合路线均可执行')

平均池化输出: (1024,)
时序注意力输出: (15, 128)
✅ 时序注意力行归一化正确


✅ 三种聚合路线均可执行


C:\Users\24260\AppData\Local\Temp\ipykernel_45132\1759029484.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ 音视频对齐（简化）

In [5]:
# 音频：过声学编码器成向量；视频：过视觉编码器；对齐 = 对比学习/拼接
n_seg = 10
aud_feat = rng.normal(0, 1, (n_seg, 512))
vid_feat = rng.normal(0, 1, (n_seg, 1024))
# 简化：投影到共同空间后算相似度
W_aud = rng.normal(0, 0.1, (512, 256)); W_vid = rng.normal(0, 0.1, (1024, 256))
a_emb = aud_feat @ W_aud; v_emb = vid_feat @ W_vid
sim_av = a_emb @ v_emb.T
print('音视频相似度矩阵:', sim_av.shape)
print('➡️ 对齐任务：音频片段与其对应视频片段相似度最高（对比学习）')

音视频相似度矩阵: (10, 10)
➡️ 对齐任务：音频片段与其对应视频片段相似度最高（对比学习）


## 5️⃣ 长视频处理（面试追问）

In [6]:
print('''长视频（>10 分钟）在 VideoLLM 里的挑战与方案：
  1) token 太多：抽关键帧 / 分段处理 / 视觉 token 压缩；
  2) 上下文超限：滑动窗口 + 分段摘要 + 记忆池；
  3) 时序理解：位置编码 / 分段重叠；
  4) 音频+字幕：多模态融合（音频/OCR 字幕辅助理解）；
  5) 评测：视频问答（VideoQA）/ 事件定位 / 摘要质量。
''')

长视频（>10 分钟）在 VideoLLM 里的挑战与方案：
  1) token 太多：抽关键帧 / 分段处理 / 视觉 token 压缩；
  2) 上下文超限：滑动窗口 + 分段摘要 + 记忆池；
  3) 时序理解：位置编码 / 分段重叠；
  4) 音频+字幕：多模态融合（音频/OCR 字幕辅助理解）；
  5) 评测：视频问答（VideoQA）/ 事件定位 / 摘要质量。



## 📝 自测清单

- [ ] 能说出视频建模三条路线及取舍
- [ ] 能手写帧采样→编码→时序聚合流程
- [ ] 能解释帧位置编码的必要性
- [ ] 能讲清音视频对齐的对比学习思路
- [ ] 能回答长视频 token 爆炸的解决方向